# Day 7 Function Guide

This notebook summarizes the files, functions, validation checks, and outputs created for Day 7 of the weather probability modeling project.

Day 7 builds the daily forecast-error target and expands it into prediction-time rows. It does **not** create hourly forecast-error targets, engineer hourly features, train models, or create bucket probabilities.

The target remains:

```python
forecast_error = actual_high - forecast_high
```

## Files Created

| File | Purpose |
| --- | --- |
| `src/target_builder.py` | Builds and validates one daily forecast-error target row per `date/location`. |
| `src/supervised_table.py` | Expands daily targets into one row per `date/location/prediction_time`. |
| `scripts/build_day7_supervised_table.py` | Runnable Day 7 pipeline that reads cleaned inputs, writes outputs, and prints validation summaries. |
| `tests/test_day7_tables.py` | Lightweight tests for Day 7 target math, row counts, timestamps, duplicate validation, and feature/target column intent. |
| `data/processed/daily_forecast_error_targets.csv` | Daily target output: one row per `date/location`. |
| `data/processed/supervised_forecast_error_rows.csv` | Supervised skeleton output: 24 hourly prediction-time rows per `date/location`. |
| `outputs/day7_targets/target_summary.csv` | Summary statistics for `forecast_error`, overall and by location/prediction time. |

## Import Setup

This cell makes imports work whether the notebook is opened from the repo root or from inside the `notebooks/` directory.

In [ ]:
from pathlib import Path
import importlib
import inspect
import sys

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## Day 7 Pipeline Command

Run the full Day 7 pipeline from the repo root with:

```powershell
python scripts\build_day7_supervised_table.py
```

## `src.target_builder`

This module owns daily target construction. It joins daily actual highs to daily forecast highs by `date/location`, computes `forecast_error`, and validates the result.

### Public Functions

| Function | Summary |
| --- | --- |
| `build_daily_forecast_error_targets(daily_df, forecasts_df)` | Standardizes date/location/high columns, joins daily actuals to daily forecasts, computes `forecast_error = actual_high - forecast_high`, and returns one row per `date/location`. |
| `validate_daily_targets(df)` | Checks required columns, missing values, duplicate `date/location` keys, target math, and suspicious target distributions. Raises `ValueError` for hard failures and warnings for distribution concerns. |

### Helper Functions

| Function | Summary |
| --- | --- |
| `_rename_first_match(df, candidates, target)` | Uses shared column-finding logic to rename the first matching candidate column to a canonical target name. |
| `_standardize_date_column(df, label)` | Finds and normalizes the daily date column to `date`. |
| `_standardize_location_column(df)` | Finds a location-like column and renames it to `location`. |
| `_fill_missing_location_values(df, label)` | Fills missing location values only when it is safe for single-location data. |
| `_single_location_value(df, label)` | Returns the single safe location value, or raises if multiple locations make inference unsafe. |
| `_align_location_columns(actual_df, forecast_df)` | Ensures both daily actual and forecast frames have compatible `location` columns. |
| `_prepare_daily_actuals(daily_df)` | Copies actual daily data, standardizes `date`, identifies `actual_high`, and coerces it numeric. |
| `_prepare_daily_forecasts(forecasts_df)` | Copies daily forecast data, standardizes `date`, identifies `forecast_high`, and coerces it numeric. |
| `_require_unique_keys(df, keys, label)` | Raises if a dataframe has duplicate key rows. |
| `_available_ordered_columns(df, columns)` | Keeps output columns in a predictable order when optional columns are present. |

## `src.supervised_table`

This module owns the Day 7 supervised row skeleton. Each daily target is repeated at every hourly prediction time from `00:00` through `23:00`.

### Constants

| Constant | Value / Purpose |
| --- | --- |
| `PREDICTION_TIMES` | `['00:00', '01:00', ..., '23:00']` |
| `TARGET_COLUMN` | `forecast_error` |
| `AUDIT_ONLY_COLUMNS` | `['actual_high']`; retained for auditing, not future model features. |
| `BASELINE_FEATURE_COLUMNS` | `['forecast_high']`; allowed later as a baseline feature. |
| `SUPERVISED_REQUIRED_COLUMNS` | Required schema for the Day 7 skeleton output. |

### Functions

| Function | Summary |
| --- | --- |
| `define_prediction_times()` | Returns the Day 7 prediction times. |
| `expand_targets_to_prediction_times(target_df, prediction_times=None)` | Cross-joins each daily target row with prediction times and adds `prediction_timestamp`. |
| `add_prediction_timestamp(df)` | Combines `date` and `prediction_time` into a parseable timestamp. |
| `validate_supervised_rows(df)` | Checks schema, prediction-time values, duplicate `date/location/prediction_time` rows, parseable timestamps, exactly 24 rows per `date/location`, and target math. |

## `scripts.build_day7_supervised_table`

This script is the one-command Day 7 runner. It creates output directories if needed, profiles cleaned inputs, builds targets, expands supervised rows, writes the three output CSVs, and prints a concise success report.

### Functions

| Function | Summary |
| --- | --- |
| `_read_required_csv(path)` | Reads a required input CSV or raises a clear `FileNotFoundError`. |
| `_date_range_text(df, column='date')` | Formats a dataframe date range for logging. |
| `_timestamp_range_text(df, column='timestamp')` | Formats a dataframe timestamp range for logging. |
| `_print_key_missing_values(df)` | Prints missing counts for important key/target columns when present. |
| `_print_duplicate_counts(df)` | Prints duplicate counts for standard daily, hourly, and supervised key sets. |
| `_print_input_profile(path)` | Prints row count, columns, date/timestamp ranges, missing values, and duplicate counts for an input file. |
| `_numeric_stat(series, stat)` | Computes a selected numeric statistic for target summaries. |
| `_quantile(series, q)` | Computes a numeric quantile for target summaries. |
| `_summary_row(df, location, prediction_time)` | Builds one summary row for a target-summary group. |
| `build_target_summary(supervised_df)` | Creates `outputs/day7_targets/target_summary.csv` content overall, by location, by prediction time, and by both. |
| `_forecast_source_warnings(forecasts_df)` | Reports proxy-forecast and missing as-of timestamp caveats. |
| `main()` | Runs the full Day 7 pipeline. |

## `tests/test_day7_tables.py`

The Day 7 tests cover the core contracts that Day 8 will depend on.

| Test | Summary |
| --- | --- |
| `test_daily_forecast_error_targets_are_one_row_per_date_location` | Verifies target construction, row count, duplicate-free daily keys, and forecast-error math. |
| `test_daily_target_validation_rejects_duplicate_keys` | Confirms duplicate daily target keys fail validation. |
| `test_supervised_rows_expand_each_daily_target_to_hourly_prediction_times` | Confirms each daily target expands to 24 hourly prediction-time rows and timestamps are correct. |
| `test_supervised_validation_rejects_duplicate_prediction_rows` | Confirms duplicate supervised keys fail validation. |
| `test_target_and_audit_columns_are_not_marked_as_baseline_features` | Documents that `forecast_error` is the target, `actual_high` is audit-only, and `forecast_high` is the allowed baseline feature. |

## Function Inventory From Source

The next cell introspects the Day 7 modules so the notebook stays useful if function signatures change later.

In [ ]:
target_builder = importlib.import_module("src.target_builder")
supervised_table = importlib.import_module("src.supervised_table")
day7_script = importlib.import_module("scripts.build_day7_supervised_table")


def function_inventory(module):
    rows = []
    for name, obj in inspect.getmembers(module, inspect.isfunction):
        if obj.__module__ != module.__name__:
            continue
        rows.append(
            {
                "module": module.__name__,
                "function": name,
                "signature": str(inspect.signature(obj)),
                "public": not name.startswith("_"),
            }
        )
    return pd.DataFrame(rows).sort_values(["public", "function"], ascending=[False, True])


pd.concat(
    [
        function_inventory(target_builder),
        function_inventory(supervised_table),
        function_inventory(day7_script),
    ],
    ignore_index=True,
)

## Output Contracts

### `data/processed/daily_forecast_error_targets.csv`

- One row per `date/location`.
- Required columns: `date`, `location`, `actual_high`, `forecast_high`, `forecast_error`.
- Optional source columns are preserved when available, such as `forecast_source`.

### `data/processed/supervised_forecast_error_rows.csv`

- One row per `date/location/prediction_time`.
- Prediction times are exactly hourly from `00:00` through `23:00`.
- Includes `prediction_timestamp` so Day 8 can attach leakage-safe features.
- Keeps `actual_high` only for auditing and keeps `forecast_error` as the target.

### `outputs/day7_targets/target_summary.csv`

- Summarizes `forecast_error` overall, by location, by prediction time, and by location/prediction time.
- Since the Day 7 target is repeated across prediction times, the prediction-time target summaries are expected to match.

## Preview Day 7 Outputs

In [ ]:
daily_targets_path = repo_root / "data" / "processed" / "daily_forecast_error_targets.csv"
supervised_rows_path = repo_root / "data" / "processed" / "supervised_forecast_error_rows.csv"
target_summary_path = repo_root / "outputs" / "day7_targets" / "target_summary.csv"

daily_targets = pd.read_csv(daily_targets_path)
supervised_rows = pd.read_csv(supervised_rows_path)
target_summary = pd.read_csv(target_summary_path)

daily_targets.head(10)

In [ ]:
supervised_rows.head(10)

In [ ]:
target_summary

## Quick Validation Checks

These checks mirror the key Day 7 success criteria without rebuilding the CSVs.

In [ ]:
daily_math_diff = (
    daily_targets["forecast_error"]
    - (daily_targets["actual_high"] - daily_targets["forecast_high"])
).abs().max()

pd.Series(
    {
        "daily_target_rows": len(daily_targets),
        "supervised_rows": len(supervised_rows),
        "daily_duplicate_date_location": int(daily_targets.duplicated(["date", "location"]).sum()),
        "supervised_duplicate_date_location_prediction_time": int(
            supervised_rows.duplicated(["date", "location", "prediction_time"]).sum()
        ),
        "prediction_times": sorted(supervised_rows["prediction_time"].unique().tolist()),
        "rows_per_date_location_values": sorted(
            supervised_rows.groupby(["date", "location"]).size().unique().tolist()
        ),
        "max_daily_forecast_error_math_diff": daily_math_diff,
    }
)

## Caveats For Later Days

- `forecast_source` is currently `open_meteo_historical_forecast`, which is proxy data rather than confirmed official NWS archived forecasts.
- The cleaned forecast rows do not include an as-of/model-run timestamp, so true point-in-time forecast availability is not fully verifiable yet.
- Day 8 should attach leakage-safe features to `supervised_forecast_error_rows.csv` using `prediction_timestamp` and the hourly actual/forecast inputs.
- `actual_high` is present in the supervised skeleton for auditing only and should be excluded from model features.